# 01. 주제 경계 후보 탐지를 위한 청킹

#### 활용 데이터 : TS3003b.txt
- 상단 3줄(회의 대주제·참여자·일시)은 청킹하지 않고 메타데이터로만 보관
- 임베딩 모델 : `BAAI/bge-m3` 토크나이저
- Chunking 방식: **발화 단위 묶기**. 발화 중간에서 자르지 않고, 발화를 순서대로 쌓다가 `CHUNK_SIZE` 토큰을 넘기 직전에 청크 닫기
- Overlap : 이전 청크 끝에서 `CHUNK_OVERLAP` 토큰 이내의 발화를 다음 청크 앞에 다시 포함
- 청크별 메타데이터: `chunk_index`(청킹 순서), `token_count`, 그리고 3단계에서 `[발화번호]` 문맥을 만들 때 쓸 `utterance_start`/`utterance_end`

#### chunk_size / chunk_overlap 선정 근거 (TS3003b 실측)

| 항목 | 값 |
|---|---|
| 발화 수 | 714개 (그중 3토큰 이하 `네.`/`음.` 등 244개) |
| 전체 토큰 (BGE-M3) | 6,148 토큰 / 약 37분 → **분당 약 166 토큰** |
| 발화당 토큰 | 평균 8.6 / 중앙값 7 / 최대 44 |


| chunk_size / overlap | 청크 수 | 청크당 발화 수 | 판단 |
|---|---|---|---|
| 128 / 32 | 64 | 14.7 | 짧은 응답 비중이 커서 임베딩이 불안정 |
| **200 / 40** | **38** | **23.1** | **약 1분 분량. 짧은 안건(공지 사항 등)도 1~2청크로 잡힘** |
| 256 / 64 | 32 | 29.9 | 사용 가능하나 짧은 안건이 앞뒤 안건과 섞이기 시작 |
| 512 / 128 | 16 | 59.1 | 한 청크에 여러 안건이 섞여 경계 위치가 흐려짐 |

→ **기본값 : `CHUNK_SIZE = 200`, `CHUNK_OVERLAP = 40`(20%)**
- 임베딩 유사도로 경계를 찾으려면 한 안건이 최소 2~3개 청크에 걸쳐야 인접 청크 간 유사도 하락이 드러남. 200토큰이면 3~5분 길이의 일반 안건이 3~5청크가 됨
- 참고 : BGE-M3 입력 한도는 8,192토큰
- 오버랩 20%는 경계 직전·직후 발화가 양쪽 청크에 함께 보이도록 하는 최소 수준. 너무 크면 인접 청크끼리 유사도가 높아져 경계 신호가 약해집니다.

In [1]:
# --------------------------------------------------------------------------------------------------
# 0. 설정 
# 경로/파라미터는 해당 셀에서만 수정
# transformers, sentence-transformers 신규 설치
# --------------------------------------------------------------------------------------------------
from pathlib import Path

# ── 경로 ──────────────────────────────────────────────
BASE_DIR = Path.cwd().parent                       # playground/chaewon
INPUT_PATH = BASE_DIR / "dataset" / "TS3003b.txt"  # 청킹할 전사문
OUTPUT_DIR = BASE_DIR / "outputs" / "chunks"       # 결과 저장 폴더 (outputs/는 .gitignore 대상)

# ── 전사문 형식 ───────────────────────────────────────
HEADER_LINES = 3            # 상단 메타데이터 줄 수 (회의 대주제 / 참여자 / 일시) → 청킹 제외
UTTERANCE_SEPARATOR = "\n"  # 청크 안에서 발화를 이어 붙일 구분자

# ── 토크나이저 / 청킹 파라미터 ─────────────────────────
TOKENIZER_NAME = "BAAI/bge-m3"  # 임베딩 모델과 같은 토크나이저로 토큰 수 계산
CHUNK_SIZE = 200                # 청크당 최대 토큰 수
CHUNK_OVERLAP = 40              # 인접 청크가 공유할 최대 토큰 수 (발화 단위로 맞춤)

# 파라미터 비교용 후보 (마지막 셀)
COMPARE_CONFIGS = [(128, 32), (160, 32), (200, 40), (256, 64), (320, 64), (512, 128)]

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1. 전사문 읽기 - 머리말, 발화 분리
# transformer 때문에 실행 조금 걸릴 수 있음 - 실행 시 조금 기다려주기!
# --------------------------------------------------------------------------------------------------
import json
from transformers import AutoTokenizer

assert 0 <= CHUNK_OVERLAP < CHUNK_SIZE, "CHUNK_OVERLAP은 0 이상, CHUNK_SIZE 미만이어야 합니다."


def load_transcript(path, header_lines):
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    header = {}
    for line in lines[:header_lines]:
        key, _, value = line.partition(":")
        header[key.strip()] = value.strip()
    utterances = [line.strip() for line in lines[header_lines:] if line.strip()]
    return header, utterances


header, utterances = load_transcript(INPUT_PATH, HEADER_LINES)
meeting_id = INPUT_PATH.stem

print("머리말 (청킹 제외):", json.dumps(header, ensure_ascii=False, indent=2))
print(f"발화 수: {len(utterances)}")
print("첫 발화 5개:", utterances[:5])

c:\Users\user\Desktop\한화내일아카데미\remeet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


머리말 (청킹 제외): {
  "회의 대주제": "Real Reaction 신제품 TV 리모컨 개발 프로젝트 2차 회의 - 기능 설계",
  "참여자": "프로젝트 매니저, 산업 디자이너, UI 디자이너, 마케팅 담당자 (4명)",
  "일시": "2004-10-11 (회의 길이 약 37분)"
}
발화 수: 714
첫 발화 5개: ['좋아요.', '다들 다시 자리 찾으셨죠? 네?', '네.', '좋네요.', '좋아요, 그럼']


In [3]:
# --------------------------------------------------------------------------------------------------
# 2. 토큰 기준 발화 단위 청킹
# --------------------------------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)


def count_tokens(text):
    # 특수 토큰(<s>, </s>)을 뺀 본문 토큰 수
    return len(tokenizer.encode(text, add_special_tokens=False))


def chunk_utterances(utterances, chunk_size, chunk_overlap, sep=UTTERANCE_SEPARATOR):
    chunks = []
    start = 0
    while start < len(utterances):
        # 1) chunk_size를 넘기 직전까지 발화를 쌓는다 (발화 1개가 chunk_size보다 길면 그 발화 하나로 청크 구성)
        end = start + 1
        while end < len(utterances) and count_tokens(sep.join(utterances[start:end + 1])) <= chunk_size:
            end += 1
        text = sep.join(utterances[start:end])
        chunks.append({
            "text": text,
            "metadata": {
                "chunk_index": len(chunks),
                "token_count": count_tokens(text),
                "utterance_start": start,    # 포함
                "utterance_end": end - 1,    # 포함
            },
        })
        if end == len(utterances):
            break
        # 2) 청크 끝에서 chunk_overlap 이내의 발화를 다음 청크 시작으로 (항상 앞으로 진행)
        next_start = end
        while next_start - 1 > start and count_tokens(sep.join(utterances[next_start - 1:end])) <= chunk_overlap:
            next_start -= 1
        start = next_start
    return chunks


chunks = chunk_utterances(utterances, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"청크 수: {len(chunks)}  (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")

청크 수: 38  (chunk_size=200, chunk_overlap=40)


In [4]:
# --------------------------------------------------------------------------------------------------
# 3. 결과 확인
# --------------------------------------------------------------------------------------------------
import pandas as pd
from IPython.display import display

chunks_df = pd.DataFrame([{**c["metadata"], "preview": c["text"][:60].replace("\n", " / ")} for c in chunks])
display(chunks_df)
display(chunks_df["token_count"].describe())

# 정합성 검사: 순서, 토큰 한도(긴 단일 발화 예외), 모든 발화 포함 여부
assert chunks_df["chunk_index"].tolist() == list(range(len(chunks)))
single = chunks_df["utterance_start"] == chunks_df["utterance_end"]
assert ((chunks_df["token_count"] <= CHUNK_SIZE) | single).all()
covered = set()
for c in chunks:
    covered.update(range(c["metadata"]["utterance_start"], c["metadata"]["utterance_end"] + 1))
assert covered == set(range(len(utterances))), "누락된 발화가 있습니다."

print("\n[chunk 0]\n" + chunks[0]["text"])
print("\n[chunk 1] (앞부분이 chunk 0 끝과 겹침)\n" + chunks[1]["text"])

,chunk_index,token_count,utterance_start,utterance_end,preview
0,0,200,0,24,"좋아요. / 다들 다시 자리 찾으셨죠? 네? / 네. / 좋네요. / 좋아요, 그럼..."
1,1,200,19,45,"네. / C동이랑 E동이에요. / 그래서 / 좀 지나면, 며칠이면 끝날 거예요. /..."
2,2,193,39,62,"음, 첫 발표부터 시작하죠. / 네. / 음, 누가 먼저 할래요? / 제가 먼저 할..."
3,3,197,59,74,"음 / 이게 제가 보기에 꽤 놀라운 결과 세 가지예요. / 음, 리모컨은 못생겼다고..."
4,4,196,73,88,"채널 선택과 볼륨 선택은 음, 1부터 10까지 중요도에서 둘 다 10점을 받았어요...."
5,5,183,87,100,"음, 새로운 기능을 쓰는 걸 좋아하지만, / 아주 까다롭기도 해요. / 돈을 쉽게 ..."
6,6,189,99,110,"음, 그리고 응답자의 약 4분의 1이 리모컨 때문에 RSI가 생긴다고 생각했어요. ..."
7,7,194,108,123,"가장 많이 쓰이니까 / 튼튼해야 해요. / 쉽게 고장 나면 안 돼요. / 음, 그리..."
8,8,189,122,155,"이건 음, 주로 16세에서 25세 사람들이었어요. / 그래도 45세까지는 여전히 가..."
9,9,199,148,169,"위쪽에 있는 거로 가세요. 슬라이드 쇼요. / 그러니까 / 아. / 어. / 아, ..."


count     38.000000
mean     193.815789
std        6.176920
min      176.000000
25%      189.250000
50%      196.000000
75%      199.000000
max      200.000000
Name: token_count, dtype: float64


[chunk 0]
좋아요.
다들 다시 자리 찾으셨죠? 네?
네.
좋네요.
좋아요, 그럼
이번이 두 번째 회의예요.
그리고 어
아직도 안 돼요?
네.
어, 이제 음, 기능 설계로 들어가요.
음, 이 단계에서 중요한 건 사용자 요구 사항, 기술적 기능 설계, 작동 설계에 대해 어, 합의를 보는 거예요. 그래야 두 번째 어, 단계로 넘어갈 수 있으니까요.
하지만 먼저 이 단계부터요.
음, 먼저 공지 사항이 있어요.
냉난방 시스템에 약간의 조정이 있어요.
그래서
또 유령 마우스가 나타났네요.
그러니까 냉난방에 약간, 약간 문제가 있을 수 있다는 거고,
그건 이 어
네.
네.
C동이랑 E동이에요.
그래서
좀 지나면, 며칠이면 끝날 거예요.
어차피 추울 거라서
필요하진 않을 것 같아요.

[chunk 1] (앞부분이 chunk 0 끝과 겹침)
네.
C동이랑 E동이에요.
그래서
좀 지나면, 며칠이면 끝날 거예요.
어차피 추울 거라서
필요하진 않을 것 같아요.
그렇죠.
그다음은 안건이에요.
자
먼저 시작.
어, 이번엔 제가 회의록을 적을게요.
어, 다들 발표를 하실 거예요. 모두요.
음, 전체 어, 발표에 40분이 있어요.
그래서 어, 발표당 7분 정도로 하는 걸 제안해요,
그다음에 새 프로젝트 요구 사항에 대해 잠깐 논의하고
어, 저한테 온 거요.
그리고 어, 넣을 조작 기능과 넣지 않을 기능을 결정하는 거예요.
그래서
전부 해서 40분이 있어요.
그러니까
음, 첫 발표부터 시작하죠.
네.
음, 누가 먼저 할래요?
제가 먼저 할게요.
좋아요.
그럼
아마 이렇게 하면 더 쉬울 것 같은데 음


In [ ]:
# --------------------------------------------------------------------------------------------------
# 4. 저장
# 현재는 각 단계별로 저장하지 않도록 함
# 만약, 단계별로 저장하고 싶다면 주석 풀고 실행하면 "outputs/chunks" 디렉토리 내 <회의ID>_chunks.json 파일로 저장됨
# utterances도 함께 저장하여 utteraces_start~utterance_end로 [발화번호] 텍스트 형태의 경계 앞뒤 문맥을 다시 만들 수 있음
# output에는 크게 아래와 같은 정보를 포함함
# > 회의ID / metadata(회의주제, 참여자, 일시) / chunking 정보 / utterances 정보(발화별 순서 인덱스) / chunks(각 청킹값)

# chunks 예)
# {'text': '좋아요.\n다들 다시 자리 찾으셨죠? 네?\n네.\n좋네요.\n좋아요, 그럼\n이번이 두 번째 회의예요.\n그리고 어\n아직도 안 돼요?\n네.\n어, 이제 음, 기능 설계로 들어가요.\n음, 이 단계에서 중요한 건 사용자 요구 사항, 기술적 기능 설계, 작동 설계에 대해 어, 합의를 보는 거예요. 그래야 두 번째 어, 단계로 넘어갈 수 있으니까요.\n하지만 먼저 이 단계부터요.\n음, 먼저 공지 사항이 있어요.\n냉난방 시스템에 약간의 조정이 있어요.\n그래서\n또 유령 마우스가 나타났네요.\n그러니까 냉난방에 약간, 약간 문제가 있을 수 있다는 거고,\n그건 이 어\n네.\n네.\nC동이랑 E동이에요.\n그래서\n좀 지나면, 며칠이면 끝날 거예요.\n어차피 추울 거라서\n필요하진 않을 것 같아요.',
#    'metadata': {'chunk_index': 0,
#     'token_count': 200,
#     'utterance_start': 0,
#     'utterance_end': 24}}
# --------------------------------------------------------------------------------------------------
output = {
    "meeting_id": meeting_id,
    "metadata": header,
    "chunking": {
        "tokenizer": TOKENIZER_NAME,
        "unit": "token (special tokens excluded)",
        "strategy": "utterance packing; never split inside an utterance",
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "separator": UTTERANCE_SEPARATOR,
        "num_utterances": len(utterances),
        "num_chunks": len(chunks),
    },
    "utterances": [{"utterance_index": i, "text": u} for i, u in enumerate(utterances)],
    "chunks": chunks,
}

output

# 저장용
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# output_path = OUTPUT_DIR / f"{meeting_id}_chunks.json"
# output_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")
# print("저장 완료:", output_path)

{'meeting_id': 'TS3003b',
 'metadata': {'회의 대주제': 'Real Reaction 신제품 TV 리모컨 개발 프로젝트 2차 회의 - 기능 설계',
  '참여자': '프로젝트 매니저, 산업 디자이너, UI 디자이너, 마케팅 담당자 (4명)',
  '일시': '2004-10-11 (회의 길이 약 37분)'},
 'chunking': {'tokenizer': 'BAAI/bge-m3',
  'unit': 'token (special tokens excluded)',
  'strategy': 'utterance packing; never split inside an utterance',
  'chunk_size': 200,
  'chunk_overlap': 40,
  'separator': '\n',
  'num_utterances': 714,
  'num_chunks': 38},
 'utterances': [{'utterance_index': 0, 'text': '좋아요.'},
  {'utterance_index': 1, 'text': '다들 다시 자리 찾으셨죠? 네?'},
  {'utterance_index': 2, 'text': '네.'},
  {'utterance_index': 3, 'text': '좋네요.'},
  {'utterance_index': 4, 'text': '좋아요, 그럼'},
  {'utterance_index': 5, 'text': '이번이 두 번째 회의예요.'},
  {'utterance_index': 6, 'text': '그리고 어'},
  {'utterance_index': 7, 'text': '아직도 안 돼요?'},
  {'utterance_index': 8, 'text': '네.'},
  {'utterance_index': 9, 'text': '어, 이제 음, 기능 설계로 들어가요.'},
  {'utterance_index': 10,
   'text': '음, 이 단계에서 중요한 건 사용자 요구 사항, 

In [ ]:
# --------------------------------------------------------------------------------------------------
# 참고) Chunk_size / Chunk_overlap 조합별 결과 비교
# chunk_size, chunk_overlap 각각 200, 40일 때 평균 토큰과 최소 토큰의 차이가 가장 적음
# --------------------------------------------------------------------------------------------------
rows = []
for size, overlap in COMPARE_CONFIGS:
    cs = chunk_utterances(utterances, size, overlap)
    tokens = [c["metadata"]["token_count"] for c in cs]
    utts = [c["metadata"]["utterance_end"] - c["metadata"]["utterance_start"] + 1 for c in cs]
    rows.append({"chunk_size": size, "chunk_overlap": overlap, "num_chunks": len(cs),
                 "avg_tokens": round(sum(tokens) / len(tokens)), "min_tokens": min(tokens),
                 "avg_utterances": round(sum(utts) / len(utts), 1)})
display(pd.DataFrame(rows))

,chunk_size,chunk_overlap,num_chunks,avg_tokens,min_tokens,avg_utterances
0,128,32,64,120,54,14.7
1,160,32,48,153,54,18.4
2,200,40,38,194,176,23.1
3,256,64,32,247,159,29.9
4,320,64,24,310,216,36.7
5,512,128,16,498,364,59.1
